In [1]:
# ============================================================
# CELL 0 — INSTALL
# ============================================================
!pip install -q "transformers>=4.46.0" accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 56.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [2]:
# ============================================================
# E5 — VLM RE-RANK (Qwen2-VL-2B)
# Notebook mới — khớp với pipeline: E0 Dense + E1 docTR + E2 RRF
# Re-rank top-10 candidates từ E0 bằng VLM confidence score
# ============================================================

# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import os, glob, json, zipfile, shutil, gc
import numpy as np
import torch
from PIL import Image
from datetime import datetime
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor


# ============================================================
# CELL 2 — SETUP: tìm artifact docTR (ưu tiên) hoặc fallback
# E5 chỉ cần 3 thứ — KHÔNG cần corpus_embeddings_shards:
#   1. e0_dense_topk_results.json  (top-10 candidates để re-rank)
#   2. corpus_id_to_idx.json       (map image_id → row index dataset)
#   3. dataset gốc (vidore/docvqa_train) để lấy ảnh PIL thật cho VLM
# ============================================================
WORK_ARTIFACT_DIR = "/kaggle/working/docvqa_colqwen2_artifacts"
RETRIEVAL_DIR     = f"{WORK_ARTIFACT_DIR}/retrieval_results"
os.makedirs(RETRIEVAL_DIR, exist_ok=True)

DATASET_NAME          = "vidore/docvqa_train"
E0_RESULTS_PATH       = f"{RETRIEVAL_DIR}/e0_dense_topk_results.json"
CORPUS_ID_TO_IDX_PATH = f"{WORK_ARTIFACT_DIR}/corpus_id_to_idx.json"

# Metrics files để bảng so sánh cuối
E0_METRICS_PATH = f"{RETRIEVAL_DIR}/e0_dense_metrics.json"
E1_METRICS_PATH = f"{RETRIEVAL_DIR}/e1_bm25_metrics.json"
E2_METRICS_PATH = f"{RETRIEVAL_DIR}/e2_rrf_metrics.json"


def find_and_copy(target_path, filename_pattern):
    if os.path.exists(target_path):
        return True
    hits = glob.glob(f"/kaggle/input/**/{filename_pattern}", recursive=True)
    if hits:
        # Ưu tiên hit nào nằm trong artifact docTR (path chứa "retest" hoặc "doctr")
        doctr_hits = [h for h in hits if "retest" in h.lower() or "doctr" in h.lower()]
        src = doctr_hits[0] if doctr_hits else hits[0]
        shutil.copy(src, target_path)
        print(f"📄 Copied: {src}")
        return True
    return False


# Giải nén artifact nếu chưa có E0 local
if not os.path.exists(E0_RESULTS_PATH):
    all_zips = [
        p for p in glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if "docvqa_colqwen2_artifacts" in os.path.basename(p)
    ]
    assert all_zips, "❌ Không tìm thấy artifact zip trong /kaggle/input"

    # Ưu tiên zip chứa ocr_text_shards_doctr (= artifact docTR)
    doctr_zips = []
    for zp in all_zips:
        with zipfile.ZipFile(zp, "r") as zf:
            if any("ocr_text_shards_doctr" in n for n in zf.namelist()):
                doctr_zips.append(zp)

    zip_path = max(doctr_zips if doctr_zips else all_zips, key=os.path.getsize)
    label = "docTR" if doctr_zips else "fallback"
    print(f"📦 Giải nén artifact ({label}): {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/kaggle/working")

# Bắt buộc phải có
assert find_and_copy(E0_RESULTS_PATH, "e0_dense_topk_results.json"), \
    "❌ Thiếu e0_dense_topk_results.json"
assert find_and_copy(CORPUS_ID_TO_IDX_PATH, "corpus_id_to_idx.json"), \
    "❌ Thiếu corpus_id_to_idx.json"

# Optional — dùng cho bảng so sánh cuối
for path, pattern in [
    (E0_METRICS_PATH, "e0_dense_metrics.json"),
    (E1_METRICS_PATH, "e1_bm25_metrics.json"),
    (E2_METRICS_PATH, "e2_rrf_metrics.json"),
]:
    ok = find_and_copy(path, pattern)
    print(f"{'✅' if ok else '⚠️ '} {pattern}")

print("\n✅ Setup xong — sẵn sàng chạy E5")


📦 Giải nén artifact (docTR): /kaggle/input/notebooks/hnganhdng/hybrid-rrf-for-bm25/docvqa_colqwen2_artifacts.zip
✅ e0_dense_metrics.json
✅ e1_bm25_metrics.json
✅ e2_rrf_metrics.json

✅ Setup xong — sẵn sàng chạy E5


In [3]:
# ============================================================
# CELL 3 — LOAD E0 RESULTS + CORPUS_ID_TO_IDX
# ============================================================
with open(E0_RESULTS_PATH, "r", encoding="utf-8") as f:
    e0_results = json.load(f)

with open(CORPUS_ID_TO_IDX_PATH, "r", encoding="utf-8") as f:
    corpus_id_to_idx = json.load(f)

print(f"E0 queries     : {len(e0_results):,}")
print(f"Corpus id→idx  : {len(corpus_id_to_idx):,} entries")
print(f"Ví dụ query[0] : {e0_results[0]['query'][:60]}...")
print(f"Candidates[0]  : {[x['image_id'][:12]+'...' for x in e0_results[0]['topk'][:3]]}")


# ============================================================
# CELL 4 — RELOAD DATASET GỐC (để lấy ảnh PIL thật)
# ============================================================
print("\nĐang load dataset gốc...")
full_ds = load_dataset(DATASET_NAME, split="train")
print(f"✅ full_ds: {len(full_ds):,} records")


def safe_load_image(img, max_side=1024):
    """Load PIL, resize cạnh dài về max_side để VLM xử lý nhanh hơn."""
    try:
        if img is None:
            raise ValueError
        img = img.convert("RGB")
        if max(img.size) > max_side:
            img.thumbnail((max_side, max_side), Image.LANCZOS)
        return img
    except Exception:
        return Image.new("RGB", (224, 224), color=(255, 255, 255))


# ============================================================
# CELL 5 — LOAD QWEN2-VL-2B
# ============================================================
VLM_MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"\nĐang load {VLM_MODEL_NAME}...")
vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_NAME,
    torch_dtype=dtype,
    device_map=None,
).to(device).eval()

vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)
print(f"✅ Model loaded  (device={device}, dtype={dtype})")

# Token id Yes / No để đọc logit sau generate 1 token duy nhất
YES_ID = vlm_processor.tokenizer.encode("Yes", add_special_tokens=False)[0]
NO_ID  = vlm_processor.tokenizer.encode("No",  add_special_tokens=False)[0]
print(f"   Token 'Yes'={YES_ID}, 'No'={NO_ID}")


# ============================================================
# CELL 6 — HÀM VLM CONFIDENCE SCORE
# ============================================================
def vlm_yes_no_score(image: Image.Image, query_text: str) -> float:
    """
    Hỏi VLM: ảnh này có chứa câu trả lời cho query không?
    Trả về: logit(Yes) - logit(No)
    Cao hơn = VLM tự tin hơn rằng ảnh này đúng.
    Chỉ generate 1 token → rất nhanh (~0.2–0.5 giây/ảnh trên T4).
    """
    question = (
        "Look at this document image carefully.\n"
        f"Does it contain the answer to this question: {query_text}\n"
        "Answer with exactly one word: Yes or No."
    )

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text":  question},
        ],
    }]

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt], images=[image],
        return_tensors="pt", padding=True,
    ).to(device)

    with torch.inference_mode():
        out = vlm_model.generate(
            **inputs,
            max_new_tokens=1,
            output_scores=True,
            return_dict_in_generate=True,
            do_sample=False,
        )

    logits = out.scores[0][0]                      # shape (vocab_size,)
    score  = float(logits[YES_ID] - logits[NO_ID]) # scalar
    return score


# ============================================================
# CELL 7 — CHẠY E5: RE-RANK TOP-10 TỪ E0 BẰNG VLM
# ============================================================
SUBSET_SIZE = 500   # 500 queries × 10 candidates = 5000 VLM calls ≈ 1.5-2h trên T4
RERANK_K    = 10    # đúng bằng top-K đã lưu trong E0
SEED        = 42

E5_RESULTS_PATH = f"{RETRIEVAL_DIR}/e5_vlm_rerank_topk_results.json"
E5_METRICS_PATH = f"{RETRIEVAL_DIR}/e5_vlm_rerank_metrics.json"
E5_PARTIAL_PATH = f"{RETRIEVAL_DIR}/e5_vlm_rerank_partial.json"

rng = np.random.default_rng(SEED)
subset_idx = sorted(rng.choice(len(e0_results), size=SUBSET_SIZE, replace=False).tolist())
print(f"📌 Subset: {SUBSET_SIZE} queries, seed={SEED}")
print(f"   Ví dụ query indices: {subset_idx[:5]}")


def compute_retrieval_metrics(results, k_values=[1, 5, 10]):
    metrics, n = {}, len(results)
    for k in k_values:
        hit = sum(
            1 for item in results
            if item["ground_truth_image_filename"] in
               [x["image_id"] for x in item["topk"][:k]]
        )
        metrics[f"Recall@{k}"] = hit / n

    rr_list = []
    for item in results:
        gt = item["ground_truth_image_filename"]
        for rank, x in enumerate(item["topk"][:10], 1):
            if x["image_id"] == gt:
                rr_list.append(1.0 / rank); break
        else:
            rr_list.append(0.0)
    metrics["MRR@10"] = float(np.mean(rr_list))

    ndcg_list = []
    for item in results:
        gt = item["ground_truth_image_filename"]
        for rank, x in enumerate(item["topk"][:10], 1):
            if x["image_id"] == gt:
                ndcg_list.append(1.0 / np.log2(rank + 1)); break
        else:
            ndcg_list.append(0.0)
    metrics["NDCG@10"] = float(np.mean(ndcg_list))
    return metrics


# Resume nếu đã có partial từ lần chạy trước (crash giữa chừng)
results, done_indices = [], set()
if os.path.exists(E5_PARTIAL_PATH):
    with open(E5_PARTIAL_PATH, encoding="utf-8") as f:
        results = json.load(f)
    done_indices = {r["query_index"] for r in results}
    print(f"⏭️  Resume: đã có {len(results)}/{SUBSET_SIZE} queries từ partial")

print(f"\n🚀 Bắt đầu E5 VLM Re-rank ({SUBSET_SIZE} queries × {RERANK_K} candidates)...")

for i in tqdm(subset_idx, desc="E5 VLM Re-rank"):
    if i in done_indices:
        continue

    e0_item    = e0_results[i]
    candidates = e0_item["topk"][:RERANK_K]

    # Score từng candidate bằng VLM
    vlm_scores = []
    for cand in candidates:
        img_id  = cand["image_id"]
        row_idx = corpus_id_to_idx[img_id]
        img     = safe_load_image(full_ds[row_idx]["image"])
        score   = vlm_yes_no_score(img, e0_item["query"])
        vlm_scores.append((img_id, score))

    # Sắp xếp lại theo VLM confidence (cao → thấp)
    vlm_scores_sorted = sorted(vlm_scores, key=lambda x: x[1], reverse=True)

    results.append({
        "query_index": i,
        "query":       e0_item["query"],
        "ground_truth_image_filename": e0_item["ground_truth_image_filename"],
        "answer":      e0_item["answer"],
        "topk": [
            {"rank": r + 1, "image_id": img_id, "score": score}
            for r, (img_id, score) in enumerate(vlm_scores_sorted)
        ],
    })

    # Save partial mỗi 25 queries
    if len(results) % 25 == 0:
        with open(E5_PARTIAL_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache()
        gc.collect()

# Save kết quả cuối
metrics = compute_retrieval_metrics(results, k_values=[1, 5, 10])
metrics_info = {
    "experiment":      "E5_Qwen2VL_2B_Rerank_of_E0_Top10",
    "vlm_model":       VLM_MODEL_NAME,
    "base_retrieval":  "E0_ColQwen2_Dense",
    "rerank_pool_size": RERANK_K,
    "subset_size":     SUBSET_SIZE,
    "seed":            SEED,
    "num_queries":     len(results),
    "created_at":      datetime.now().isoformat(),
    "metrics":         metrics,
}

with open(E5_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(E5_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_info, f, ensure_ascii=False, indent=2)

print("\n✅ E5 VLM Re-rank finished!")
for k, v in metrics.items():
    print(f"   {k}: {v:.4f}")


# ============================================================
# CELL 8 — SO SÁNH FAIR: E5 vs E0 TRÊN CÙNG SUBSET 500 QUERIES
# KHÔNG so với E0 full 2000 — sẽ lệch phân phối
# ============================================================
e0_subset = [e0_results[i] for i in subset_idx]
e0_sub_m  = compute_retrieval_metrics(e0_subset)

print("\n========== SO SÁNH TRÊN CÙNG SUBSET (n=500) ==========")
print(f"{'Metric':<12} {'E0 (subset)':>12} {'E5 VLM':>12}   Delta")
print("-" * 52)
for m in ["Recall@1", "Recall@5", "Recall@10", "MRR@10", "NDCG@10"]:
    v0, v5 = e0_sub_m[m], metrics[m]
    delta = v5 - v0
    sign = "+" if delta >= 0 else ""
    print(f"{m:<12} {v0:>12.4f} {v5:>12.4f}   {sign}{delta:.4f}")
print("=======================================================")

# Tham chiếu thêm E1, E2 trên full 2000 (để viết báo cáo)
print("\n(Tham chiếu full 2000 queries — không so trực tiếp với E5)")
for label, path in [("E0 (full)", E0_METRICS_PATH),
                    ("E1 docTR",  E1_METRICS_PATH),
                    ("E2 RRF",   E2_METRICS_PATH)]:
    if os.path.exists(path):
        with open(path) as f:
            mm = json.load(f)["metrics"]
        print(f"   {label:<12} Recall@1={mm['Recall@1']:.4f}  MRR@10={mm['MRR@10']:.4f}")


# ============================================================
# CELL 9 — ZIP LẠI
# ============================================================
zip_path = "/kaggle/working/docvqa_colqwen2_artifacts.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive("/kaggle/working/docvqa_colqwen2_artifacts", "zip",
                    "/kaggle/working", "docvqa_colqwen2_artifacts")
print(f"\n✅ Đã zip: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    for fname in ["e5_vlm_rerank_topk_results.json", "e5_vlm_rerank_metrics.json"]:
        found = any(n.endswith(fname) for n in names)
        print(f"{'✅' if found else '❌'} {fname}")

print("\n⚠️  Upload version mới này trước khi sang End-to-End + ANLS!")

E0 queries     : 2,000
Corpus id→idx  : 10,189 entries
Ví dụ query[0] : What is the Submitting Office No.?
Provide a short and direc...
Candidates[0]  : ['b241d26b1be7...', 'ef3cd0ab94ff...', '0423a5cff2ed...']

Đang load dataset gốc...


README.md:   0%|          | 0.00/516 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

data/train-00000-of-00057.parquet:   0%|          | 0.00/26.8M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded  (device=cuda, dtype=torch.bfloat16)
   Token 'Yes'=9454, 'No'=2753
📌 Subset: 500 queries, seed=42
   Ví dụ query indices: [9, 11, 12, 17, 26]

🚀 Bắt đầu E5 VLM Re-rank (500 queries × 10 candidates)...


E5 VLM Re-rank:   0%|          | 0/500 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



✅ E5 VLM Re-rank finished!
   Recall@1: 0.2660
   Recall@5: 0.5060
   Recall@10: 0.6100
   MRR@10: 0.3696
   NDCG@10: 0.4269

========== SO SÁNH TRÊN CÙNG SUBSET (n=500) ==========
Metric        E0 (subset)       E5 VLM   Delta
----------------------------------------------------
Recall@1           0.3460       0.2660   -0.0800
Recall@5           0.5320       0.5060   -0.0260
Recall@10          0.6100       0.6100   +0.0000
MRR@10             0.4249       0.3696   -0.0552
NDCG@10            0.4690       0.4269   -0.0421

(Tham chiếu full 2000 queries — không so trực tiếp với E5)
   E0 (full)    Recall@1=0.3775  MRR@10=0.4580
   E1 docTR     Recall@1=0.1555  MRR@10=0.1907
   E2 RRF       Recall@1=0.3385  MRR@10=0.4088

✅ Đã zip: /kaggle/working/docvqa_colqwen2_artifacts.zip (1498.2 MB)
✅ e5_vlm_rerank_topk_results.json
✅ e5_vlm_rerank_metrics.json

⚠️  Upload version mới này trước khi sang End-to-End + ANLS!
